In [ ]:
# ============================================================
# MFC3: Enhancing Summarization Efficiency
# Text Summarization using T5 — Group D16
# Team Members: Indraneel R,Abhishek R,D Rishikesh,M Aryan
# Amrita Vishwa Vidyapeetham

# ── SECTION 1: Install Dependencies ─────────────────────────

import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

def pip_uninstall(pkg):
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", pkg])

print("🔧 Cleaning conflicting installations...")

# Upgrade pip
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

# Remove conflicting packages
pip_uninstall("transformers")
pip_uninstall("accelerate")
pip_uninstall("datasets")

print("📦 Installing compatible versions...")

pip_install("transformers==4.37.2")
pip_install("accelerate==0.27.2")
pip_install("datasets==2.18.0")
pip_install("sentencepiece")
pip_install("evaluate")
pip_install("rouge_score")
pip_install("tensorboard")

# ── SECTION 2: Imports ───────────────────────────────────────
import os
import glob
import pprint
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import evaluate

from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from datasets import load_dataset

pp = pprint.PrettyPrinter(indent=2)

print("=" * 60)
print("  MFC3 — Text Summarization using T5")
print("  Group D1 | Amrita Vishwa Vidyapeetham")
print("=" * 60)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n✅ Device: {device}")

# ── SECTION 3: Configuration ─────────────────────────────────
MODEL       = "t5-base"
BATCH_SIZE  = 4
NUM_PROCS   = 1
EPOCHS      = 10
MAX_LENGTH  = 512
GEN_LENGTH  = 80
NUM_BEAMS   = 4
LR          = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
OUT_DIR     = "/content/results_t5base"
os.makedirs(OUT_DIR, exist_ok=True)

# ── SECTION 4: Load Dataset ───────────────────────────────────
print("\n📦 Loading dataset...")
raw_dataset  = load_dataset("gopalkalpande/bbc-news-summary", split="train")
full_dataset = raw_dataset.train_test_split(test_size=0.2, shuffle=True, seed=42)
dataset_train = full_dataset["train"]
dataset_valid = full_dataset["test"]

# ── SECTION 6: Tokenizer ──────────────────────────────────────
tokenizer = T5Tokenizer.from_pretrained(MODEL, legacy=False)

def preprocess_function(examples):
    inputs = ["summarize: " + article for article in examples["Articles"]]
    targets = examples["Summaries"]

    model_inputs = tokenizer(inputs, max_length=MAX_LENGTH, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=GEN_LENGTH, truncation=True, padding="max_length")

    label_ids = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = label_ids
    return model_inputs

tokenized_train = dataset_train.map(preprocess_function, batched=True, num_proc=NUM_PROCS)
tokenized_valid = dataset_valid.map(preprocess_function, batched=True, num_proc=NUM_PROCS)

# ── SECTION 7: Load Model ─────────────────────────────────────
model = T5ForConditionalGeneration.from_pretrained(MODEL).to(device)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)

# ── SECTION 8: ROUGE ─────────────────────────────────────────
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {k: round(v, 4) for k, v in result.items()}

# ── SECTION 9: Training Arguments ────────────────────────────
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    # ✅ FIXED LINE
    evaluation_strategy="epoch",

    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=True,
    generation_max_length=GEN_LENGTH,
    generation_num_beams=NUM_BEAMS,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ── SECTION 10: Train ─────────────────────────────────────────
trainer.train()

# ── SECTION 11: Save ─────────────────────────────────────────
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

# ── SECTION 12: Evaluate ─────────────────────────────────────
results = trainer.evaluate()
print(results)

🔧 Cleaning conflicting installations...
📦 Installing compatible versions...


In [ ]:
# ================================
# 🚀 MFC3 GRADIO UI
# ================================

!pip install -q gradio transformers sentencepiece

import os
import torch
import gradio as gr
from transformers import T5Tokenizer, T5ForConditionalGeneration

# ================================
# 🔧 LOAD MODEL (SAFE LOADING)
# ================================

MODEL_PATH = "mfc3_final_model"  # ⚠️ removed ./ to avoid HF error

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("🔄 Loading model...")

if os.path.exists(MODEL_PATH):
    print("📂 Loading from local trained model...")
    tokenizer = T5Tokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
    model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH, local_files_only=True).to(device)
else:
    print("⚠️ Local model not found. Using pretrained t5-base...")
    tokenizer = T5Tokenizer.from_pretrained("t5-base")
    model = T5ForConditionalGeneration.from_pretrained("t5-base").to(device)

model.eval()
print("✅ Model Ready!")

# ================================
# 🧠 SUMMARIZATION FUNCTION
# ================================
def summarize_text(text, beams, max_len):
    if not text.strip():
        return "⚠️ Please enter some text."

    inputs = tokenizer(
        "summarize: " + text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_len,
            num_beams=beams,
            early_stopping=True
        )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# ================================
# 🎨 GRADIO INTERFACE
# ================================
interface = gr.Interface(
    fn=summarize_text,
    inputs=[
        gr.Textbox(lines=12, placeholder="Paste your article here...", label="📰 Input Article"),
        gr.Slider(1, 10, value=5, step=1, label="🔍 Beam Search"),
        gr.Slider(20, 150, value=64, step=5, label="📏 Summary Length")
    ],
    outputs=gr.Textbox(label="✨ Generated Summary"),
    title="📄 MFC3 Text Summarization (T5)",
    description="Fine-tuned T5 summarizer with beam search decoding",
)

# ================================
# ▶️ LAUNCH
# ================================
interface.launch(share=True)

🔄 Loading model...
⚠️ Local model not found. Using pretrained t5-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Model Ready!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6e61ca7452a52b0cdb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
